In this first step, I basically set up everything I needed to start working on the project. I installed all the important libraries like pandas, numpy, matplotlib, and seaborn, which help with handling data and creating graphs. I also used Prophet for forecasting future sales and scikit-learn for things like anomaly detection and evaluating my model. After installing, I imported all these libraries into my code so I could actually use them. This step is important because without these tools, I wouldn’t be able to analyze or visualize the data properly.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import IsolationForest


In [3]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import IsolationForest

In [4]:
mydf = pd.read_csv(r"C:\Users\mkhan\OneDrive\Desktop\Dashboard Tab\Sample - Superstore.csv", encoding="latin1")

mydf.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


Here, I loaded the dataset into Python so I could start working with it. The dataset had order details like sales, profit, dates, etc. I converted the date columns into proper datetime format because I needed them later for time-based analysis like monthly trends and forecasting. I also handled missing values by replacing them with zero to avoid errors during calculations. This step is really important because if the data is messy or inconsistent, the whole analysis can go wrong.

In [5]:
# LOAD & CLEAN DATA
# Convert dates
mydf['Order Date'] = pd.to_datetime(mydf['Order Date'])
mydf['Ship Date'] = pd.to_datetime(mydf['Ship Date'])

# Handle missing values
mydf.fillna(0, inplace=True)

mydf.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


This is honestly one of the most important parts of my project. Instead of just using the raw data, I created new columns that help me understand the business better. For example, I extracted year, month, and quarter from the order date so I could analyze trends over time. I calculated profit margin to see how much profit is made from sales, and I created a discount impact column to understand how discounts affect revenue. I also added order value to analyze how much customers spend per purchase. On top of that, I created previous month sales and rolling average sales to track trends and smooth out fluctuations. This step makes the project look more advanced because it shows deeper thinking, not just basic analysis.

In [6]:
# FEATURE ENGINEERING
# Time Features
mydf['Year'] = mydf['Order Date'].dt.year
mydf['Month'] = mydf['Order Date'].dt.month
mydf['Quarter'] = mydf['Order Date'].dt.quarter

# Pricing Features
mydf['Profit Margin'] = (mydf['Profit'] / mydf['Sales']) * 100
mydf['Discount Impact'] = mydf['Discount'] * mydf['Sales']

# Customer Behavior
mydf['Order Value'] = mydf['Sales'] / mydf['Quantity']

# Lag Features (for trend understanding)
mydf['Prev Month Sales'] = mydf['Sales'].shift(1)

# Rolling average (smoothing)
mydf['Rolling Sales'] = mydf['Sales'].rolling(window=3).mean()

In this section, I started analyzing the data from a business perspective. First, I grouped the data by category and calculated total sales, total profit, average profit margin, and average discount. This helped me understand which categories are performing well and which ones might be struggling. Then, I calculated monthly sales so I could see how sales change over time. This kind of analysis is useful because businesses usually want to know where they are making money and where they are losing it.

In [7]:
#BUSINESS ANALYSIS
# Category-Level Insights
category_analysis = mydf.groupby('Category').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Profit Margin': 'mean',
    'Discount': 'mean'
}).reset_index()

print(category_analysis)

          Category        Sales       Profit  Profit Margin  Discount
0        Furniture  741999.7953   18451.2728       3.878353  0.173923
1  Office Supplies  719047.0320  122490.8008      13.803029  0.157285
2       Technology  836154.0330  145454.9481      15.613805  0.132323


In [8]:
#Monthly Sales
monthly_sales = mydf.groupby(
    pd.Grouper(key='Order Date', freq='ME')
)['Sales'].sum().reset_index()

Here, I used Prophet to predict future sales. I first adjusted the dataset into the format Prophet needs, where the date column is called “ds” and the sales column is called “y”. Then I trained the model using past sales data. After that, I generated future dates for the next six months and predicted sales for those months. This part is really useful because it shows how data can be used to plan ahead, like estimating future demand or setting business targets.

In [10]:
# ===============================
# SALES FORECASTING
# ===============================

from prophet import Prophet

# Prepare data
prophet_df = monthly_sales.rename(columns={
    'Order Date': 'ds',
    'Sales': 'y'
})

# Create model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

# Fit model
model.fit(prophet_df)

# Create future dates
future = model.make_future_dataframe(periods=6, freq='M')

# Predict
forecast = model.predict(future)

# Show results
print(forecast[['ds', 'yhat']].tail())

21:14:17 - cmdstanpy - INFO - Chain [1] start processing
21:14:18 - cmdstanpy - INFO - Chain [1] done processing


           ds          yhat
49 2018-02-28  35661.188232
50 2018-03-31  71959.640532
51 2018-04-30  55942.164642
52 2018-05-31  56193.017015
53 2018-06-30  57387.968259


C:\Users\mkhan\anaconda3\envs\indata_ip25\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(


After building the forecasting model, I plotted the results to actually see what’s going on. The first graph shows the predicted sales along with past sales, so it’s easy to understand the trend. The second graph breaks down things like trend and seasonality, which helps explain why the model is predicting certain patterns. Visualization makes everything much clearer compared to just looking at numbers.

In [ ]:
#Plot
model.plot(forecast)
plt.title("Sales Forecast")
plt.show()

model.plot_components(forecast)
plt.show()

In this step, I checked how accurate my model is. I compared the actual sales values with the predicted values and calculated MAE and RMSE. These metrics basically tell me how far off my predictions are from the real values. This is important because I don’t want to just build a model — I want to make sure it actually works well.

In [ ]:
#MODEL EVALUATION
merged = prophet_mydf.merge(
    forecast[['ds', 'yhat']],
    on='ds',
    how='left'
)

mae = mean_absolute_error(merged['y'], merged['yhat'])
rmse = np.sqrt(mean_squared_error(merged['y'], merged['yhat']))

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))

In this section, I tried to find unusual patterns in the data. I used the Isolation Forest model to detect anomalies, which are basically unexpected spikes or drops in sales. The model labels normal points as 1 and unusual ones as -1. Then I plotted the results and highlighted the anomalies in red. This is useful because businesses can use it to detect issues like sudden demand changes or possible errors in data.

In [ ]:
#ANOMALY DETECTION
iso_model = IsolationForest(
    contamination=0.05,
    random_state=42
)

monthly_sales['Anomaly'] = iso_model.fit_predict(
    monthly_sales[['Sales']]
)

In [ ]:
#Plot Anomalies
plt.figure(figsize=(12,6))

plt.plot(monthly_sales['Order Date'], monthly_sales['Sales'])

anomalies = monthly_sales[monthly_sales['Anomaly'] == -1]

plt.scatter(
    anomalies['Order Date'],
    anomalies['Sales'],
    color='red'
)

plt.title("Anomaly Detection")
plt.show()

This is probably the most important part from a business point of view. I used the data to find products with low profit margins, which means the company isn’t making much money on them. I also calculated how much revenue is being lost because of discounts. Then I created something called a revenue score to identify which products have the best potential. These insights can help businesses adjust pricing, reduce unnecessary discounts, and improve overall profit.

In [ ]:
# PRICING OPTIMIZATION INSIGHTS
low_margin = mydf[mydf['Profit Margin'] < 10]
print(low_margin[['Category', 'Sales', 'Profit Margin']])

In [ ]:
#High Discount Loss
discount_loss = mydf.groupby('Category')['Discount Impact'].sum()
print(discount_loss)

In [ ]:
#Revenue Opportunity Score
mydf['Revenue Score'] = mydf['Sales'] * (1 - mydf['Discount'])

top_opportunities = mydf.sort_values(by='Revenue Score', ascending=False)

Finally, I exported all the processed data into CSV files so I could use them in tools like Tableau or Power BI. I saved different outputs like forecast results, anomaly detection results, category analysis, and the cleaned dataset. This step is important because it connects the Python analysis to dashboards, which are easier to present and share.

In [ ]:
# EXPORT DATA (FOR DASHBOARD)
forecast.to_csv("forecast_results.csv", index=False)
monthly_sales.to_csv("anomaly_results.csv", index=False)
category_analysis.to_csv("category_analysis.csv", index=False)
mydf.to_csv("cleaned_full_data.csv", index=False)

So overall, what I did in this project is I started with raw sales data, cleaned it, and created new features to make it more useful. Then I analyzed the data to understand business performance, built a model to predict future sales, and checked how accurate it is. I also detected unusual patterns and created pricing insights to help improve profit. At the end, I prepared everything for dashboards so it can be presented in a clear and professional way. This project shows how data can be used to solve real business problems and support better decisions.